# Transfer Learning: DQN con pesos pre-entrenados

En lugar de entrenar DQN desde cero (1M+ steps sin completar el nivel),
descargamos un modelo **PPO** pre-entrenado que **ya resolvió Mario 1-1** y
transferimos los pesos de la CNN (feature extractor) a nuestro DQN.

**Estrategia:**
1. Descargar modelo PPO de HuggingFace (tsilva/Level1-1)
2. Extraer pesos de la NatureCNN (3 capas conv compartidas)
3. Inyectarlos en un DQN nuevo con nuestro entorno env-v2
4. Fine-tune ~300-500k steps con LR reducido

**¿Por qué funciona?** Ambos modelos usan la misma arquitectura NatureCNN de SB3
(conv 32×8×8/4 → 64×4×4/2 → 64×3×3/1 → FC 512) y el mismo espacio de
observación (4×84×84 grayscale). La CNN aprende features visuales genéricas
(bordes, plataformas, enemigos) que son reutilizables entre PPO y DQN.



In [ ]:
# 1) GPU
!nvidia-smi
import torch; print("CUDA:", torch.cuda.is_available())

Wed Oct 28 23:50:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2) Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 3) Código del proyecto desde GitHub
REPO_URL = "https://github.com/David-AGL/mario-bros-reinforcement-learning.git"
BRANCH = "main"

import os
if not os.path.exists("/content/mario/.git"):
    !git clone -b {BRANCH} {REPO_URL} /content/mario
else:
    !git -C /content/mario pull origin {BRANCH}
%cd /content/mario
!git log --oneline -1

Cloning into '/content/mario'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 192 (delta 76), reused 173 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 301.93 KiB | 10.06 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/mario
b118165 (HEAD -> main, origin/main, origin/HEAD) implementacion de interfaz con streamlit


In [ ]:
# 4) Dependencias
# Desinstalamos librerías conflictivas preinstaladas en Colab que fuerzan NumPy 2.x
!pip uninstall -y -q tensorflow tensorflow-cpu jax jaxlib opencv-python opencv-contrib-python 2>/dev/null
!pip install -q "numpy<2" "gym==0.26.2" "nes-py==8.2.1" "gym-super-mario-bros==7.4.0" \
    "gymnasium==0.29.1" "stable-baselines3==2.3.2" "opencv-python-headless<=4.10.0.84" psutil tensorboard \
    huggingface_sb3 "huggingface_hub<1.0.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optax 0.2.8 requires jax>=0.5.3, which is not installed.
optax 0.2.8 requires jaxlib>=0.5.3, which is not installed.
flax 0.11.2 requires jax>=0.6.0, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not install

In [ ]:
!pip uninstall -y -q tensorflow tensorflow-cpu jax jaxlib 2>/dev/null
!python -c "from stable_baselines3 import DQN, PPO; print('SB3 OK')"

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
SB3 OK


In [ ]:
# 5) Smoke test del entorno
!python -W ignore scripts/smoke_test_env.py

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
SuperMarioBros-1-1-v0: OK | episodios=6 max_x=1411 FPS(agent steps)=77
SuperMarioBros-2-1-v0: OK | episodios=14 max_x=722 FPS(agent steps)=87
SuperMarioBros-3-1-v0: OK | episodios=16 max_x=1086 FPS(agent steps)=91
SuperMarioBros-1-2-v0: OK | episodios=16 max_x=916 FPS(agent steps)=90
No-ops reproducibles con la misma semilla: [1, 1, 1]
SMOKE TEST OK


## Paso 1: Descargar modelo PPO pre-entrenado de HuggingFace

Usamos el modelo de `tsilva` que fue entrenado con PPO + CnnPolicy (NatureCNN)
en SuperMarioBros Level 1-1 con observaciones (4, 84, 84) grayscale y
SIMPLE_MOVEMENT (7 acciones).

In [ ]:
import zipfile
import torch

# 1. Abrir el .zip y extraer el state_dict de PyTorch directamente
print(f"Abriendo {checkpoint_path}...")
with zipfile.ZipFile(checkpoint_path, 'r') as archive:
    print("Archivos dentro del ZIP:", archive.namelist())
    # Extraemos el archivo de pesos de la política
    with archive.open('policy.pth') as pth_file:
        state_dict = torch.load(pth_file, map_location="cpu")

print(f"¡Pesos de PyTorch extraídos con éxito! Total llaves: {len(state_dict)}")

# 2. Filtrar únicamente las capas de la NatureCNN (features_extractor)
cnn_weights = {}
for k, v in state_dict.items():
    if "features_extractor" in k:
        # Renombramos quitando el prefijo si es necesario para que encaje directo
        clean_key = k.replace("features_extractor.", "")
        cnn_weights[clean_key] = v
        print(f"  Capas CNN: {clean_key} -> shape {v.shape}")

print("\n¡Pesos convolucionales listos para inyectar en tu DQN!")

Abriendo /root/.cache/huggingface/hub/models--tsilva--Level1-1_stable-baselines3-ppo_7d9b9131/snapshots/83af48e18749ad223073d1c944000e801b96917e/model.zip...
Archivos dentro del ZIP: ['data', 'pytorch_variables.pth', 'policy.pth', 'policy.optimizer.pth', '_stable_baselines3_version', 'system_info.txt']
✅ ¡Pesos de PyTorch extraídos con éxito! Total llaves: 28
  Capas CNN: cnn.0.weight -> shape torch.Size([32, 4, 8, 8])
  Capas CNN: cnn.0.bias -> shape torch.Size([32])
  Capas CNN: cnn.2.weight -> shape torch.Size([64, 32, 4, 4])
  Capas CNN: cnn.2.bias -> shape torch.Size([64])
  Capas CNN: cnn.4.weight -> shape torch.Size([64, 64, 3, 3])
  Capas CNN: cnn.4.bias -> shape torch.Size([64])
  Capas CNN: linear.0.weight -> shape torch.Size([512, 3136])
  Capas CNN: linear.0.bias -> shape torch.Size([512])
  Capas CNN: pi_cnn.0.weight -> shape torch.Size([32, 4, 8, 8])
  Capas CNN: pi_cnn.0.bias -> shape torch.Size([32])
  Capas CNN: pi_cnn.2.weight -> shape torch.Size([64, 32, 4, 4])
  Cap

## Paso 2: Inspeccionar y verificar compatibilidad de la CNN

In [ ]:
# Inspeccionar la arquitectura del features_extractor (NatureCNN)
if ppo_model is not None:
    print("=== NatureCNN del modelo PPO pre-entrenado ===")
    fe = ppo_model.policy.features_extractor
    print(fe)
    print(f"\nParámetros CNN: {sum(p.numel() for p in fe.parameters()):,}")
    print("\nPesos por capa:")
    for name, param in fe.named_parameters():
        print(f"  {name}: {param.shape}")

## Paso 3: Crear DQN y transferir pesos de la CNN

Creamos un DQN nuevo con nuestro entorno env-v2 y copiamos los pesos
convolucionales del PPO pre-entrenado. Solo las capas de la CNN se transfieren;
las capas FC del Q-network se inicializan de cero (porque PPO tiene actor+critic
y DQN tiene Q-values, son salidas diferentes).

In [ ]:
# ─── Configuración de la corrida ───
SEED = 1
TOTAL_STEPS = 500_000          # menos que antes: la CNN ya sabe ver
RUN_NAME = "dqn_transfer_s1"

DRIVE = "/content/drive/MyDrive/mario"
MODELS_ROOT = f"{DRIVE}/models"
LOGS_ROOT = "/content/runs"
LOGS_DRIVE = f"{DRIVE}/runs"

# Hiperparámetros de fine-tuning (ajustados para transfer)
LEARNING_RATE = 5e-5            # más bajo que 1e-4: no destruir los pesos transferidos
BUFFER_SIZE = 100_000
LEARNING_STARTS = 10_000        # menos warm-up: la CNN ya extrae features útiles
BATCH_SIZE = 32
EXPLORATION_INITIAL_EPS = 0.3   # empezar con menos exploración (no desde 1.0)
EXPLORATION_FINAL_EPS = 0.02    # más greedy al final
EXPLORATION_FRACTION = 0.3      # explorar 30% de los steps
TARGET_UPDATE_INTERVAL = 10_000
CHECKPOINT_FREQ = 50_000        # guardar más seguido para detectar el mejor

# Congelar CNN o no
FREEZE_CNN = False  # False = fine-tune completo; True = solo entrena capas FC

EXTRA = "--optimize-memory-usage"

print(f"Plan: {TOTAL_STEPS:,} steps, LR={LEARNING_RATE}, eps={EXPLORATION_INITIAL_EPS}→{EXPLORATION_FINAL_EPS}")
print(f"CNN congelada: {FREEZE_CNN}")

Plan: 500,000 steps, LR=5e-05, eps=0.3→0.02
CNN congelada: False


In [ ]:
import sys, json, torch
import numpy as np
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList

sys.path.insert(0, "/content/mario")
from env import ENV_VERSION, make_env
from dqn.train import MarioDQNCallback

# ─── 1. Crear entorno (misma config que corridas anteriores) ───
def _make():
    return Monitor(make_env("SuperMarioBros-1-1-v0", noop_max=30))

venv = DummyVecEnv([_make])
venv.seed(SEED)

# ─── 2. Crear DQN nuevo ───
dqn_model = DQN(
    "CnnPolicy", venv,
    learning_rate=LEARNING_RATE,
    buffer_size=BUFFER_SIZE,
    learning_starts=LEARNING_STARTS,
    batch_size=BATCH_SIZE,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=TARGET_UPDATE_INTERVAL,
    exploration_fraction=EXPLORATION_FRACTION,
    exploration_initial_eps=EXPLORATION_INITIAL_EPS,
    exploration_final_eps=EXPLORATION_FINAL_EPS,
    max_grad_norm=10.0,
    optimize_memory_usage=True,
    replay_buffer_kwargs=dict(handle_timeout_termination=False),
    tensorboard_log=LOGS_ROOT + "/dqn",
    seed=SEED,
    device="auto",
    verbose=1,
)

print("\n=== DQN antes del transfer ===")
print(f"features_extractor: {dqn_model.policy.features_extractor}")
print(f"q_net: {dqn_model.policy.q_net}")

/usr/local/lib/python3.13/dist-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(


Using cuda device

=== DQN antes del transfer ===
features_extractor: None
q_net: QNetwork(
  (features_extractor): NatureCNN(
    (cnn): Sequential(
      (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
      (3): ReLU()
      (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
      (5): ReLU()
      (6): Flatten(start_dim=1, end_dim=-1)
    )
    (linear): Sequential(
      (0): Linear(in_features=3136, out_features=512, bias=True)
      (1): ReLU()
    )
  )
  (q_net): Sequential(
    (0): Linear(in_features=512, out_features=7, bias=True)
  )
)


In [ ]:
# ─── 3. TRANSFER: copiar pesos de la CNN del PPO al DQN ───

if 'cnn_weights' in globals() and cnn_weights:
    try:
        # Filtrar solo las llaves que pertenecen a la NatureCNN estándar (cnn.X y linear.X)
        # Descartamos las de pi_cnn, vf_cnn, pi_linear y vf_linear
        filtered_cnn_weights = {
            k: v for k, v in cnn_weights.items()
            if k.startswith("cnn.") or k.startswith("linear.")
        }

        # En DQN de SB3, el extractor de características está dentro de q_net y q_net_target
        dqn_model.policy.q_net.features_extractor.load_state_dict(filtered_cnn_weights)
        dqn_model.policy.q_net_target.features_extractor.load_state_dict(filtered_cnn_weights)
        print("🚀 Pesos transferidos correctamente al DQN (dentro de q_net y q_net_target). ¡Listo para entrenar!")
        print(f"   Capas cargadas: {list(filtered_cnn_weights.keys())}")
    except Exception as e:
        print(f"Error al transferir los pesos directamente: {e}")
else:
    print("No se encontró la variable cnn_weights con los pesos extraídos.")

🚀 Pesos transferidos correctamente al DQN (dentro de q_net y q_net_target). ¡Listo para entrenar!
   Capas cargadas: ['cnn.0.weight', 'cnn.0.bias', 'cnn.2.weight', 'cnn.2.bias', 'cnn.4.weight', 'cnn.4.bias', 'linear.0.weight', 'linear.0.bias']


In [ ]:
# ─── 4. (Opcional) Congelar la CNN ───
if FREEZE_CNN:
    for param in dqn_model.policy.features_extractor.parameters():
        param.requires_grad = False
    for param in dqn_model.policy.q_net_target.features_extractor.parameters():
        param.requires_grad = False

    trainable = sum(p.numel() for p in dqn_model.policy.parameters() if p.requires_grad)
    total = sum(p.numel() for p in dqn_model.policy.parameters())
    print(f"CNN congelada. Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
else:
    trainable = sum(p.numel() for p in dqn_model.policy.parameters() if p.requires_grad)
    total = sum(p.numel() for p in dqn_model.policy.parameters())
    print(f"Fine-tune completo. Todos los parámetros entrenables: {trainable:,} / {total:,}")

Fine-tune completo. Todos los parámetros entrenables: 3,375,438 / 3,375,438


## Paso 4: Entrenar (fine-tune)

In [ ]:
import os, threading, time, subprocess

model_dir = Path(MODELS_ROOT) / "dqn" / RUN_NAME
log_dir = Path(LOGS_ROOT) / "dqn" / RUN_NAME
model_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
os.makedirs(LOGS_DRIVE, exist_ok=True)

# Guardar config
config = {
    "experiment": "transfer_learning",
    "source_model": REPO_ID if ppo_model else "none",
    "source_algo": "PPO",
    "target_algo": "DQN",
    "transfer_type": "CNN_features_extractor",
    "freeze_cnn": FREEZE_CNN,
    "total_steps": TOTAL_STEPS,
    "learning_rate": LEARNING_RATE,
    "exploration_initial_eps": EXPLORATION_INITIAL_EPS,
    "exploration_final_eps": EXPLORATION_FINAL_EPS,
    "exploration_fraction": EXPLORATION_FRACTION,
    "buffer_size": BUFFER_SIZE,
    "learning_starts": LEARNING_STARTS,
    "checkpoint_freq": CHECKPOINT_FREQ,
    "seed": SEED,
    "env_version": ENV_VERSION,
}
(model_dir / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

# Sync logs a Drive cada 5 min
_stop = False
def _sync():
    while not _stop:
        subprocess.run(["rsync", "-a", f"{LOGS_ROOT}/", f"{LOGS_DRIVE}/"])
        time.sleep(300)
threading.Thread(target=_sync, daemon=True).start()
print("Sync de logs activado.")

{
  "experiment": "transfer_learning",
  "source_model": "none",
  "source_algo": "PPO",
  "target_algo": "DQN",
  "transfer_type": "CNN_features_extractor",
  "freeze_cnn": false,
  "total_steps": 500000,
  "learning_rate": 5e-05,
  "exploration_initial_eps": 0.3,
  "exploration_final_eps": 0.02,
  "exploration_fraction": 0.3,
  "buffer_size": 100000,
  "learning_starts": 10000,
  "checkpoint_freq": 50000,
  "seed": 1,
  "env_version": "env-v2"
}
Sync de logs activado.


In [ ]:
# ─── ENTRENAR ───
callbacks = CallbackList([
    CheckpointCallback(
        save_freq=CHECKPOINT_FREQ,
        save_path=str(model_dir),
        name_prefix=f"dqn_s{SEED}",
        save_replay_buffer=False,
    ),
    MarioDQNCallback(log_dir / "episodes.csv", verbose=1),
])

print(f"\nIniciando fine-tune: {TOTAL_STEPS:,} steps")
print(f"   Checkpoints cada {CHECKPOINT_FREQ:,} steps en {model_dir}")
print(f"   Logs en {log_dir}")

dqn_model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=callbacks,
    tb_log_name=RUN_NAME,
    reset_num_timesteps=True,
    log_interval=10,
)

# Guardar modelo final
final_path = model_dir / f"dqn_s{SEED}_final"
dqn_model.save(str(final_path))
print(f"\n Modelo final guardado: {final_path}.zip")

Se truncaron las últimas líneas 5000 del resultado de transmisión.
|    fps              | 74       |
|    time_elapsed     | 3257     |
|    total_timesteps  | 241587   |
| train/              |          |
|    learning_rate    | 5e-05    |
|    loss             | 7.13     |
|    n_updates        | 57896    |
----------------------------------
----------------------------------
| mario/              |          |
|    flag_get         | 0        |
|    max_x            | 684      |
| rollout/            |          |
|    ep_len_mean      | 119      |
|    ep_rew_mean      | 766      |
|    exploration_rate | 0.02     |
| time/               |          |
|    episodes         | 2360     |
|    fps              | 74       |
|    time_elapsed     | 3275     |
|    total_timesteps  | 242882   |
| train/              |          |
|    learning_rate    | 5e-05    |
|    loss             | 12       |
|    n_updates        | 58220    |
----------------------------------
-----------------------

## Paso 5: Evaluar todos los checkpoints

Evaluamos cada checkpoint con el mismo protocolo de siempre
(epsilon-greedy 0.05, seeds 100-104, noop_max=30).

In [ ]:
import glob

# Encontrar todos los checkpoints
checkpoints = sorted(glob.glob(str(model_dir / "dqn_s*_steps.zip")))
final = model_dir / f"dqn_s{SEED}_final.zip"
if final.exists():
    checkpoints.append(str(final))

print(f"Checkpoints encontrados: {len(checkpoints)}")
for cp in checkpoints:
    print(f"  {cp}")

Checkpoints encontrados: 11
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_100000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_150000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_200000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_250000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_300000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_350000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_400000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_450000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_500000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_50000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_final.zip


In [ ]:
import re

results = []

for cp_path in checkpoints:
    # Extraer steps del nombre
    m = re.search(r"(\d+)_steps", cp_path)
    if m:
        steps_k = int(m.group(1)) // 1000
        tag = f"dqn_transfer_{steps_k}k"
    else:
        tag = "dqn_transfer_final"
        steps_k = TOTAL_STEPS // 1000

    print(f"\n{'='*60}")
    print(f"Evaluando: {tag} ({cp_path})")
    print(f"{'='*60}")

    # Usamos la ruta absoluta de evaluate.py dentro del repositorio clonado
    !python /content/mario/evaluate.py --algo dqn \
        --model "{cp_path}" \
        --level SuperMarioBros-1-1-v0 \
        --seeds 100-104 \
        --mode stochastic \
        --epsilon 0.05 \
        --noop-max 30 \
        --out results/val \
        --tag "{tag}"


Evaluando: dqn_transfer_100k (/content/drive/MyDrive/mario/models/dqn/dqn_transfer_s1/dqn_s1_100000_steps.zip)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
[1/5] seed=100 noop=23 max_x=313 completed=0 steps=30 reward=249.0
[2/5] seed=101 noop=9 max_x=313 completed=0 steps=30 reward=249.0
[3/5] seed=102 noop=13 max_x=823 completed=0 steps

In [ ]:
# ─── COMPARACIÓN ESTRUCTURADA DE RESULTADOS ───
import pandas as pd
import glob
import json
from pathlib import Path

# Obtener archivos de evaluación únicos
val_files = sorted(glob.glob("results/val/*.csv")) + sorted(glob.glob("/content/mario/results/val/*.csv"))
val_files = list(dict.fromkeys(val_files))

# Categorizar de manera limpia
baselines = []
transfer_basic = []
transfer_advanced = []

for f in val_files:
    try:
        json_file = Path(f).with_suffix(".json")
        if json_file.exists():
            data = json.loads(json_file.read_text())
            metrics = data.get("summary", data)

            tag = json_file.stem
            record = {
                "Checkpoint": tag,
                "Max X Mean": metrics.get("max_x_mean", 0.0),
                "Progress": metrics.get("progress_mean", 0.0),
                "Success Rate": metrics.get("completion_rate", 0.0)
            }

            if "adv_phase" in tag:
                transfer_advanced.append(record)
            elif "transfer" in tag:
                transfer_basic.append(record)
            else:
                baselines.append(record)
    except Exception as e:
        pass

# Crear DataFrames y formatear
def display_ranking(title, data_list):
    print(f"\n{title}")
    print("=" * 75)
    if data_list:
        df_temp = pd.DataFrame(data_list).sort_values("Max X Mean", ascending=False)
        print(df_temp.to_string(index=False, formatters={
            'Max X Mean': '{:,.1f}'.format,
            'Progress': '{:.2%}'.format,
            'Success Rate': '{:.1%}'.format
        }))
    else:
        print("No se encontraron resultados para esta categoría.")
    print("=" * 75)

# Mostrar tablas por separado para evitar mezclas
display_ranking("CATEGORÍA 1: Baselines (Entrenamiento desde cero)", baselines)
display_ranking("CATEGORÍA 2: Transfer Learning Básico (CNN Libre desde el inicio)", transfer_basic)
display_ranking("CATEGORÍA 3: Transfer Learning Avanzado por Fases (CNN Temporalmente Congelada)", transfer_advanced)


🟢 CATEGORÍA 1: Baselines (Entrenamiento desde cero)
                              Checkpoint Max X Mean Progress Success Rate
                   dqn_v2_explore05_900k    1,113.8   35.24%         0.0%
                             dqn_v2_700k      936.6   29.63%         0.0%
                             dqn_v2_600k      863.2   27.31%         0.0%
                             dqn_v2_300k      805.8   25.49%         0.0%
                             dqn_v2_400k      690.6   21.85%         0.0%
                   dqn_v2_explore05_600k      677.4   21.43%         0.0%
                             dqn_v2_200k      659.4   20.86%         0.0%
  random_1-1_uniform_random_seeds100-104      631.6   19.98%         0.0%
                   dqn_v2_explore05_700k      553.0   17.49%         0.0%
                             dqn_v2_800k      538.2   17.03%         0.0%
dqn_1-1_epsilon_greedy_0.05_seeds100-104      505.4      NaN         0.0%
                   dqn_v2_explore05_400k      459.2   14.53

## Notas importantes

### ¿Por qué transfer learning ayuda aquí?
- La **NatureCNN** (3 capas conv) aprende a detectar features visuales genéricas:
  bordes de plataformas, enemigos, pipes, bloques, huecos.
- Estas features son **agnósticas al algoritmo** (PPO o DQN las necesita por igual).
- Al transferir la CNN, tu DQN **arranca sabiendo VER**, y solo necesita aprender
  los **Q-values** (qué acciones tomar en cada estado).

### Qué esperar
- El agente debería **superar el 35% de progreso** mucho antes de 500k steps.
- Si la CNN pre-entrenada incluyó HUD crop pero la tuya no: los pesos aún ayudan
  pero el fine-tune debe adaptarse al input ligeramente diferente.
- Si completion sigue en 0% después de 500k steps, probar:
  1. Descongelar la CNN (si estaba congelada)
  2. Reducir aún más el LR a 2.5e-5
  3. Aumentar a 1M steps

### Si el modelo de HuggingFace usa `stable-retro` en vez de `gym-super-mario-bros`
El modelo `tsilva` usa `stable-retro` que tiene un preprocesamiento ligeramente
diferente (HUD crop). Los pesos de la CNN siguen siendo transferibles porque la
estructura es idéntica (NatureCNN de SB3). El fine-tune se encargará de ajustar
los filtros a tu input específico.

###  Propuesta de Mejora: Entrenamiento por Fases (Feature Extraction + Fine-Tuning Avanzado)

Para evitar que los gradientes iniciales ruidosos destruyan los excelentes filtros visuales de la CNN pre-entrenada, utilizaremos un Callback personalizado para descongelar la red progresivamente.

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class UnfreezeCNNCallback(BaseCallback):
    def __init__(self, unfreeze_at_step=50000, lr_after_unfreeze=1e-5, verbose=0):
        super().__init__(verbose)
        self.unfreeze_at_step = unfreeze_at_step
        self.lr_after_unfreeze = lr_after_unfreeze
        self.unfrozen = False

    def _on_step(self) -> bool:
        if not self.unfrozen and self.num_timesteps >= self.unfreeze_at_step:
            self.unfrozen = True
            # Descongelar capas en q_net y q_net_target
            for param in self.model.policy.q_net.features_extractor.parameters():
                param.requires_grad = True
            for param in self.model.policy.q_net_target.features_extractor.parameters():
                param.requires_grad = True

            # Ajustar la tasa de aprendizaje para el fine-tune fino
            self.model.learning_rate = self.lr_after_unfreeze
            # Re-crear el optimizador con los nuevos parámetros entrenables
            self.model.policy.set_training_mode(True)
            self.model.policy.optimizer = self.model.policy.optimizer_class(
                self.model.policy.parameters(),
                lr=self.lr_after_unfreeze,
                **self.model.policy.optimizer_kwargs
            )

            if self.verbose > 0:
                print(f"\n[Callback] ¡Paso {self.num_timesteps}! CNN descongelada. Ajustando LR a {self.lr_after_unfreeze}")
        return True

Ahora podemos configurar este entrenamiento alternativo para superar el 40% de progreso:

In [ ]:
# 1. Crear el modelo DQN avanzado corrigiendo el conflicto de parámetros
dqn_advanced = DQN(
    "CnnPolicy", venv,
    learning_rate=1e-4, # LR inicial para entrenar la cabeza lineal rápidamente
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.3,
    exploration_initial_eps=0.3,
    exploration_final_eps=0.02,
    optimize_memory_usage=False, # Evita conflicto con el manejo de timeouts del buffer
    replay_buffer_kwargs=dict(handle_timeout_termination=False),
    tensorboard_log=LOGS_ROOT + "/dqn",
    seed=SEED,
    device="auto"
)

# 2. Cargar pesos de la CNN
if 'filtered_cnn_weights' in globals() and filtered_cnn_weights:
    dqn_advanced.policy.q_net.features_extractor.load_state_dict(filtered_cnn_weights)
    dqn_advanced.policy.q_net_target.features_extractor.load_state_dict(filtered_cnn_weights)

    # 3. CONGELAR de inicio para Fase 1
    for param in dqn_advanced.policy.q_net.features_extractor.parameters():
        param.requires_grad = False
    for param in dqn_advanced.policy.q_net_target.features_extractor.parameters():
        param.requires_grad = False
    print(" Pesos cargados e inicializados en modo CONGELADO para Fase 1.")
else:
    print(" No se encontraron pesos filtrados. Asegúrate de haber ejecutado la extracción de pesos.")

✅ Pesos cargados e inicializados en modo CONGELADO para Fase 1.


In [ ]:
# ─── ENTRENAMIENTO AVANZADO POR FASES ───
import os
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList

# Definir rutas específicas para esta corrida avanzada
RUN_NAME_ADVANCED = "dqn_advanced_phase_tuning"
model_dir_adv = Path(MODELS_ROOT) / "dqn" / RUN_NAME_ADVANCED
log_dir_adv = Path(LOGS_ROOT) / "dqn" / RUN_NAME_ADVANCED
model_dir_adv.mkdir(parents=True, exist_ok=True)
log_dir_adv.mkdir(parents=True, exist_ok=True)

# Configurar callbacks
checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=str(model_dir_adv),
    name_prefix="dqn_adv",
    save_replay_buffer=False
)

# Inicializar nuestro callback dinámico de descongelamiento progresivo
# Descongelará la CNN en el paso 50,000 y bajará el LR a 1e-5 para un ajuste ultrafino
unfreeze_callback = UnfreezeCNNCallback(
    unfreeze_at_step=50_000,
    lr_after_unfreeze=1e-5,
    verbose=1
)

mario_callback = MarioDQNCallback(log_dir_adv / "episodes.csv", verbose=1)

callbacks_adv = CallbackList([checkpoint_callback, unfreeze_callback, mario_callback])

print(f"Iniciando entrenamiento optimizado en 2 fases...")
print(f"Fase 1: CNN Congelada (0 -> 50,000 steps) | LR = 1e-4")
print(f"Fase 2: CNN Descongelada (50,000 -> 500,000 steps) | LR = 1e-5")

dqn_advanced.learn(
    total_timesteps=500_000,
    callback=callbacks_adv,
    tb_log_name=RUN_NAME_ADVANCED,
    reset_num_timesteps=True,
    log_interval=100
)

# Guardar el modelo avanzado final
dqn_advanced.save(str(model_dir_adv / "dqn_adv_final"))
print("🎉 ¡Entrenamiento completado y modelo guardado exitosamente!")

🚀 Iniciando entrenamiento optimizado en 2 fases...
🔹 Fase 1: CNN Congelada (0 -> 50,000 steps) | LR = 1e-4
🔸 Fase 2: CNN Descongelada (50,000 -> 500,000 steps) | LR = 1e-5
[mem] steps=10000 RSS=5.26 GB
[mem] steps=20000 RSS=5.82 GB
[mem] steps=30000 RSS=6.39 GB
[mem] steps=40000 RSS=6.95 GB

[Callback] ¡Paso 50000! CNN descongelada. Ajustando LR a 1e-05
[mem] steps=50000 RSS=7.52 GB
[mem] steps=60000 RSS=8.09 GB
[mem] steps=70000 RSS=8.65 GB
[mem] steps=80000 RSS=9.22 GB
[mem] steps=90000 RSS=9.78 GB
[mem] steps=100000 RSS=10.31 GB
[mem] steps=110000 RSS=10.30 GB
[mem] steps=120000 RSS=10.30 GB
[mem] steps=130000 RSS=10.30 GB
[mem] steps=140000 RSS=10.30 GB
[mem] steps=150000 RSS=10.30 GB
[mem] steps=160000 RSS=10.29 GB
[mem] steps=170000 RSS=10.29 GB
[mem] steps=180000 RSS=10.29 GB
[mem] steps=190000 RSS=10.29 GB
[mem] steps=200000 RSS=10.27 GB
[mem] steps=210000 RSS=10.25 GB
[mem] steps=220000 RSS=10.25 GB
[mem] steps=230000 RSS=10.25 GB
[mem] steps=240000 RSS=10.25 GB
[mem] steps=25

## Paso 6: Evaluar Checkpoints del Entrenamiento Avanzado por Fases

Buscamos los nuevos checkpoints guardados en `dqn_advanced_phase_tuning` y ejecutamos la rutina de evaluación para compararlos en el ranking general.

In [ ]:
import glob
import re
from pathlib import Path

# 1. Encontrar los nuevos checkpoints del modelo avanzado
advanced_checkpoints = sorted(glob.glob(str(model_dir_adv / "dqn_adv_*_steps.zip")))
final_adv = model_dir_adv / "dqn_adv_final.zip"
if final_adv.exists():
    advanced_checkpoints.append(str(final_adv))

print(f"Checkpoints avanzados encontrados: {len(advanced_checkpoints)}")
for cp in advanced_checkpoints:
    print(f"  {cp}")

# 2. Ejecutar la evaluación de forma automática para cada uno
for cp_path in advanced_checkpoints:
    # Extraer steps del nombre para ponerle una etiqueta adecuada
    m = re.search(r"(\d+)_steps", cp_path)
    if m:
        steps_k = int(m.group(1)) // 1000
        tag = f"dqn_adv_phase_{steps_k}k"
    else:
        tag = "dqn_adv_phase_final"

    print(f"\n{'='*60}")
    print(f"Evaluando: {tag} ({cp_path})")
    print(f"{'='*60}")

    # Usamos la ruta absoluta de evaluate.py dentro del repositorio clonado
    !python /content/mario/evaluate.py --algo dqn \
        --model "{cp_path}" \
        --level SuperMarioBros-1-1-v0 \
        --seeds 100-104 \
        --mode stochastic \
        --epsilon 0.05 \
        --noop-max 30 \
        --out results/val \
        --tag "{tag}"

Checkpoints avanzados encontrados: 11
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_100000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_150000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_200000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_250000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_300000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_350000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_400000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_450000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_500000_steps.zip
  /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_50000_steps.zip
  /content/drive/MyDrive/mario/mo


##  Grabación de Video de la Partida del Agente Campeón

Usaremos el wrapper `VecVideoRecorder` de Stable-Baselines3 para capturar un episodio completo jugado por el modelo campeón de la fase avanzada (`dqn_adv_450000_steps` o el modelo que prefieras de tus checkpoints guardados, en este caso cargaremos el de 450,000 pasos que obtuvo el récord del 47% en promedio y Récord Individual (Mejor Partida): 78% , 2,466 px (Seed 104).



### 📺 Reproducir el Video dentro de Google Colab

Ejecuta la siguiente celda para incrustar el reproductor multimedia HTML5 y ver jugar a tu agente entrenado.


In [15]:
import os, sys, cv2, subprocess
from pathlib import Path
import numpy as np
from stable_baselines3 import DQN
from IPython.display import HTML
from base64 import b64encode

# 1. Configuración de Entorno y Rutas
if "/content/mario" not in sys.path:
    sys.path.insert(0, "/content/mario")
from env import make_env

MODEL_PATH = "/content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_450000_steps.zip"
RAW_VIDEO = "/content/mario_raw.mp4"
WEB_VIDEO = "/content/drive/MyDrive/mario/videos/mario_record_78pct_h264.mp4"
Path(WEB_VIDEO).parent.mkdir(parents=True, exist_ok=True)

# 2. Inicializar Entorno y Modelo (Seed 104 - Récord 2,466 px)
EVAL_SEED = 104
env = make_env("SuperMarioBros-1-1-v0", noop_max=30, render_mode="rgb_array")
obs, info = env.reset(seed=EVAL_SEED)
print(f" Semilla: {EVAL_SEED} | No-ops usados: {info.get('noop_used')} (debe ser 21)")

print(f"Cargando modelo: {MODEL_PATH}")
model = DQN.load(MODEL_PATH, device="cpu")
rng = np.random.default_rng([EVAL_SEED, 1])

# 3. Grabar la Partida
first_frame = env.render()
h, w, _ = first_frame.shape
writer = cv2.VideoWriter(RAW_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), 15, (w * 2, h * 2))

steps = 0
max_x = 0
total_reward = 0.0

print(" Grabando partida...")
while True:
    frame = env.render()
    frame_scaled = cv2.resize(frame, (w * 2, h * 2), interpolation=cv2.INTER_NEAREST)
    writer.write(cv2.cvtColor(frame_scaled, cv2.COLOR_RGB2BGR))

    # Epsilon 0.05 idéntico al protocolo de evaluación
    if rng.random() < 0.05:
        action = int(rng.integers(7))
    else:
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)

    obs, reward, terminated, truncated, info = env.step(action)
    steps += 1
    total_reward += float(reward)
    max_x = max(max_x, int(info.get("x_pos", 0)))

    if terminated or truncated:
        break

# Congelar los últimos frames para observar el desenlace
for _ in range(20):
    writer.write(cv2.cvtColor(frame_scaled, cv2.COLOR_RGB2BGR))

writer.release()
env.close()

print(f" Grabación finalizada: Max X={max_x} px | Pasos={steps} | Recompensa={total_reward}")

# 4. Convertir a H.264 para compatibilidad web en Chrome
print(" Codificando video a formato H.264 compatible con navegador...")
cmd = f"ffmpeg -y -loglevel error -i {RAW_VIDEO} -vcodec libx264 -crf 20 -pix_fmt yuv420p {WEB_VIDEO}"
subprocess.run(cmd, shell=True, check=True)

# 5. Reproducir directamente en la celda
print(f" Video listo y guardado en Drive: {WEB_VIDEO}")
video_bytes = open(WEB_VIDEO, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(video_bytes).decode()

HTML(f"""
<div style="text-align: center;">
  <h3>Super Mario Bros DQN - Mejor Partida (Progreso: {max_x}/3161 px - 78%)</h3>
  <video width="640" height="480" controls autoplay>
        <source src="{data_url}" type="video/mp4">
  </video>
</div>
""")

 Semilla: 104 | No-ops usados: 21 (debe ser 21)
Cargando modelo: /content/drive/MyDrive/mario/models/dqn/dqn_advanced_phase_tuning/dqn_adv_450000_steps.zip
 Grabando partida...
 Grabación finalizada: Max X=2466 px | Pasos=412 | Recompensa=2329.0
 Codificando video a formato H.264 compatible con navegador...
 Video listo y guardado en Drive: /content/drive/MyDrive/mario/videos/mario_record_78pct_h264.mp4
